# Прогнозирование стоимости аренды жилья Airbnb в Стамбуле

**Дорофеев Иван Денисович, группа 3823Б1ФИ1**

Данные: [Inside Airbnb](https://insideairbnb.com/get-the-data/), Istanbul, снимок от 30 июня 2026 г. (файл `listings.csv.gz`, детальные данные об объявлениях).

## 0. Постановка задачи

Каждая строка таблицы — объявление о сдаче жилья на Airbnb в Стамбуле: квартира, комната в квартире, номер в отеле и т.п. Для объявления известны
его местоположение (район, координаты), тип жилья, вместимость, число спален, кроватей и ванных, список удобств, правила бронирования (минимальный и максимальный срок),
занятость календаря, отзывы и оценки гостей, сведения о хозяине (как давно на сайте, сколько у него объявлений, есть ли статус суперхоста),
а также текстовые поля: название объявления, описание и рассказ хозяина о себе.

**Задача — регрессия:** по характеристикам объявления предсказать **цену за одну ночь** (`price`, в турецких лирах).

Физический смысл: хозяин, который выставляет новое объявление, хочет понять, какую цену назначить, чтобы не продешевить и не отпугнуть гостей;
гость или аналитик хочет понимать, адекватна ли цена конкретного предложения и от чего она в основном зависит (район, размер жилья, тип, рейтинг…).

Особенности данных (по ним датасет и выбирался):
- есть **пропущенные значения** (часть столбцов вообще пустая, у объявлений без отзывов нет оценок, не у всех указано число кроватей/ванных и т.д.);
- признаки **разнородные**: числовые, категориальные, бинарные, даты, список удобств, произвольный текст;
- много **выбросов** (цены от десятков лир до миллиона за ночь, минимальный срок аренды до 1000 ночей и т.п.);
- есть **текстовые признаки** (`name`, `description`, `host_about`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
sklearn.__version__

## 1. Читаем данные

Файл с данными лежит в моём репозитории (копия снимка с сайта Inside Airbnb, чтобы ноутбук всегда работал на одних и тех же данных).

In [ ]:
url = 'https://raw.githubusercontent.com/4elodoy-Molovek/ML/main/data/listings_istanbul.csv.gz'
# Исходный источник:
# url = 'https://data.insideairbnb.com/turkey/marmara/istanbul/2026-06-30/data/listings.csv.gz'
data_raw = pd.read_csv(url)
data_raw.shape

In [ ]:
data_raw.head()

In [ ]:
data_raw.info()

В таблице 26 631 объявление и 90 столбцов. Описание всех столбцов есть в [словаре данных Inside Airbnb](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4).

Целевая переменная `price` записана строкой вида `$1,356.82` (знак доллара — особенность выгрузки, на самом деле цены в лирах, это видно по полю `price_quote_raw`). Переведём её в число:

In [ ]:
data_raw['price'] = data_raw['price'].str.replace(r'[\$,]', '', regex=True).astype(float)
data_raw['price'].describe()

In [ ]:
data_raw['price_quote_raw'].dropna().iloc[5][:400]

### Какие столбцы не годятся в качестве признаков

1. **Полностью пустые** столбцы — в этом снимке их 13 (например, `host_since`, `neighborhood_overview`, `instant_bookable`).
2. **Идентификаторы и ссылки** (`id`, `listing_url`, `host_id`, `picture_url`, …) и **служебная информация о выгрузке** (`scrape_id`, `last_scraped`, `calendar_last_scraped`) — никакой информации о цене они не несут.
3. **Утечка целевой переменной.** Столбцы `price_quote_*` — это результат запроса цены на конкретные даты, `price_quote_price_per_night` в точности совпадает с `price`. `estimated_revenue_l365d` — оценка годовой выручки, которая считается как цена × число занятых ночей. Если оставить эти столбцы, модель будет «подсматривать» ответ.
4. `host_name` — имя хозяина, `license` — номер лицензии (почти у всех уникален). Из `license` возьмём только факт наличия лицензии.

In [ ]:
empty_cols = [c for c in data_raw.columns if data_raw[c].isna().all()]
empty_cols

In [ ]:
(data_raw['price_quote_price_per_night'] == data_raw['price']).mean(), data_raw[['price', 'price_quote_price_per_night']].corr().iloc[0, 1]

### Предобработка «нестандартных» признаков

Некоторые признаки нельзя сразу подать в модель, из них нужно получить числа:
- `bathrooms_text` (например, `'1.5 shared baths'`): извлекаем число ванных (им же заполним пропуски в `bathrooms`) и признак «ванная общая»;
- `amenities` — список удобств в формате JSON: считаем количество удобств и делаем бинарные признаки для нескольких самых важных (кондиционер, кухня, стиральная машина, лифт, бесплатная парковка, вид на море и т.д.);
- даты `first_review`, `last_review` → сколько дней прошло до даты выгрузки (30.06.2026);
- `license` → есть ли лицензия;
- тексты `name`, `description`, `host_about` пока просто оставим как есть (ниже построим по ним простые числовые признаки — длины, а в п. 10 попробуем TF-IDF).

In [ ]:
import json

data = data_raw.copy()

# Число ванных и признак "общая ванная"
bath_num = data['bathrooms_text'].str.extract(r'(\d+\.?\d*)')[0].astype(float)
bath_num[data['bathrooms_text'].str.contains('half', case=False, na=False)] = 0.5
data['bathrooms'] = data['bathrooms'].fillna(bath_num)
data['bath_shared'] = data['bathrooms_text'].str.contains('shared', case=False).astype(float)  # NaN останется NaN

# Удобства
amen = data['amenities'].apply(json.loads)
data['amenities_count'] = amen.apply(len)
amen_lower = amen.apply(lambda lst: ' | '.join(lst).lower())
key_amenities = {
    'am_air_conditioning': 'air conditioning',
    'am_kitchen': 'kitchen',
    'am_washer': 'washer',
    'am_elevator': 'elevator',
    'am_free_parking': 'free parking',
    'am_pool': 'pool',
    'am_sea_view': 'sea view',
    'am_balcony': 'balcony',
    'am_workspace': 'dedicated workspace',
    'am_self_checkin': 'self check-in',
}
for col, word in key_amenities.items():
    data[col] = amen_lower.str.contains(word, regex=False).astype(int)

# Даты -> число дней до даты выгрузки
snapshot = pd.Timestamp('2026-06-30')
for col in ['first_review', 'last_review']:
    data['days_since_' + col] = (snapshot - pd.to_datetime(data[col])).dt.days

data['has_license'] = data['license'].notna().astype(int)

# Длины текстов (для пропущенного текста длина 0)
for col in ['name', 'description', 'host_about']:
    data[col + '_len'] = data[col].fillna('').str.len()

data[['bathrooms_text', 'bathrooms', 'bath_shared', 'amenities_count', 'am_air_conditioning', 'am_sea_view',
      'days_since_first_review', 'has_license', 'description_len']].head()

In [ ]:
drop_cols = empty_cols + [
    # идентификаторы, ссылки, служебная информация
    'id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'picture_url', 'host_id', 'host_url',
    'host_profile_id', 'host_profile_url', 'host_picture_url', 'calendar_last_scraped',
    # утечка целевой переменной
    'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price',
    'price_quote_price_per_night', 'price_quote_raw', 'estimated_revenue_l365d',
    # уже переработаны в числовые признаки
    'bathrooms_text', 'amenities', 'first_review', 'last_review', 'license',
    # прочее
    'host_name', 'host_location',
]
data = data.drop(columns=drop_cols)
data.shape

Бинарные признаки `t`/`f` переведём в 0/1, категориальные — в тип `category`:

In [ ]:
binary_cols = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'has_availability']
for col in binary_cols:
    data[col] = data[col].map({'t': 1, 'f': 0})

cat_cols = ['neighbourhood_cleansed', 'property_type', 'room_type']
for col in cat_cols:
    data[col] = data[col].astype('category')

text_cols = ['name', 'description', 'host_about']

data[cat_cols].describe()

### Удаление невалидных записей

Как и на лекции, сначала отфильтруем записи, которые **физически или логически невозможны** — это не борьба с выбросами, а очистка ошибок. Такие правила можно применить ко всем данным до разбиения (и точно так же они применялись бы к новым объявлениям):
- объявления без цены — для них нечего предсказывать (их около 10%);
- цена меньше 150 ₺ за ночь (≈ 3–4 евро) — явная ошибка или «заглушка»; цена больше 100 000 ₺ (≈ 2 000 евро) за ночь — встречаются единичные объявления с ценой до 1.7 млн ₺, это тоже ошибки/нерабочие объявления;
- число спален, кроватей или ванных больше 20 (есть объявления с «50 спальнями»).

In [ ]:
data['price'].quantile([0, .001, .005, .01, .05, .25, .5, .75, .95, .99, .995, .999, 1])

In [ ]:
n0 = len(data)
data = data[data['price'].notna()]
data = data[data['price'].between(150, 100_000)]
for col in ['bedrooms', 'beds', 'bathrooms']:
    data = data[~(data[col] > 20)]          # NaN оставляем — их заполним позже
data = data.reset_index(drop=True)
n0, len(data), round(1 - len(data) / n0, 3)

## 2. Разбиваем данные на обучающую и тестовую выборки

Разобьём данные в пропорции 3:1. Всё дальнейшее (статистики, границы для выбросов, медианы для заполнения пропусков, параметры нормализации) будем считать **только по обучающей выборке**, чтобы информация из теста никак не попадала в модель.

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=['price'])
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
N_train, d = X_train.shape
N_test, _ = X_test.shape
N_train, N_test, d

## 3. Визуализация и основные характеристики

Для удобства соберём обучающую выборку вместе с ценой в одну таблицу.

In [ ]:
train = X_train.copy()
train['price'] = y_train
train['log_price'] = np.log(y_train)

In [ ]:
train.describe().T

In [ ]:
train.describe(include=['category']).T

### Распределение цены

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train['price'], bins=100, ax=axes[0])
axes[0].set_title('Цена за ночь, ₺')
sns.histplot(train['log_price'], bins=60, kde=True, ax=axes[1])
axes[1].set_title('Логарифм цены')
pass

In [ ]:
train['price'].agg(['mean', 'median', 'std', 'skew']), train['log_price'].agg(['mean', 'median', 'std', 'skew'])

Распределение цены сильно скошено вправо: среднее (≈ 4 200 ₺) заметно больше медианы (≈ 2 900 ₺), есть длинный «хвост» дорогих объявлений, коэффициент асимметрии ≈ 5.6. Логарифм цены распределён гораздо симметричнее (асимметрия ≈ −0.4).

Поэтому **предсказывать будем логарифм цены** `log(price)`. Ошибка в логарифмах соответствует относительной ошибке в цене: промах на 1000 ₺ для объявления за 1500 ₺ — это очень плохо, а для объявления за 30 000 ₺ — нормально. Если обучаться на самой цене, то ошибку (MSE) будут определять в основном несколько очень дорогих объявлений.

На гистограмме логарифма видны **два «горба»**: основной в районе 8–8.5 (≈ 3 000–5 000 ₺) и второй в районе 6–7 (≈ 400–1 100 ₺). Разберёмся, откуда второй.

### Краткосрочная и долгосрочная аренда

In [ ]:
train['minimum_nights'].value_counts().head(8)

In [ ]:
train['long_term'] = train['minimum_nights'] >= 28
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(x='log_price', hue='long_term', data=train, bins=60, ax=axes[0])
sns.boxplot(x='has_license', y='log_price', hue='long_term', data=train, ax=axes[1])
pass

In [ ]:
pd.crosstab(train['has_license'], train['long_term'], margins=True)

In [ ]:
train.groupby('long_term')['price'].agg(['count', 'median'])

Вот и объяснение второго горба. Почти у трети объявлений минимальный срок аренды — **100 ночей** (а у большинства остальных — 1–3 ночи). В Турции для краткосрочной сдачи жилья (на срок до 100 дней) нужно разрешение, и объявления без лицензии выставляются с минимальным сроком 100 ночей, то есть фактически как долгосрочная аренда. Цена за ночь у таких объявлений — это, по сути, месячная арендная плата, делённая на 30, поэтому она в несколько раз ниже (медиана ≈ 690 ₺ против ≈ 4 300 ₺ у краткосрочных).

Видно, что признаки `minimum_nights` и `has_license` почти однозначно определяют, к какой группе относится объявление, — это, скорее всего, будут самые важные признаки.

### Цена в зависимости от типа жилья и района

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.countplot(y='room_type', data=train, order=train['room_type'].value_counts().index, ax=axes[0])
sns.boxplot(x='log_price', y='room_type', data=train, order=train['room_type'].value_counts().index, ax=axes[1])
pass

In [ ]:
train.groupby('room_type', observed=True)['price'].agg(['count', 'median', 'mean'])

Две трети объявлений — квартиры и дома целиком (`Entire home/apt`), около трети — отдельные комнаты. Медианная цена отдельной комнаты почти в 2 раза ниже, чем у жилья целиком. Номера в отелях (`Hotel room`) — малочисленная, но самая дорогая группа. Немного неожиданно, что общие комнаты (`Shared room`, в основном места в хостелах) по медиане дороже отдельных комнат — но их всего около сотни, и среди отдельных комнат много долгосрочных объявлений с низкой ценой за ночь (см. выше).

In [ ]:
top_neigh = train['neighbourhood_cleansed'].value_counts().index[:15]
plt.figure(figsize=(10, 6))
order = train[train['neighbourhood_cleansed'].isin(top_neigh)].groupby('neighbourhood_cleansed', observed=True)['log_price'].median().sort_values(ascending=False).index
sns.boxplot(x='log_price', y='neighbourhood_cleansed', data=train[train['neighbourhood_cleansed'].isin(top_neigh)], order=order)
plt.title('15 самых крупных районов')
pass

Больше всего объявлений в центральных районах: Бейоглу (Таксим, Галата), Фатих (исторический центр, Султанахмет), Шишли, Кадыкёй, Бешикташ. Медианные цены по районам заметно различаются. Самый дорогой из крупных районов — Шиле (курортный район на берегу Чёрного моря, там сдают дома и виллы), за ним центральные Бейоглу, Шишли, Бешикташ. Дешевле всего — спальные районы Кягытхане, Малтепе, Эсенъюрт. Почти во всех районах очень большой разброс цен — опять же из-за смеси посуточных и долгосрочных объявлений.

### Географическое расположение

In [ ]:
plt.figure(figsize=(12, 8))
sample = train.sample(8000, random_state=1)
sc = plt.scatter(sample['longitude'], sample['latitude'], c=sample['log_price'], cmap='viridis', s=4, alpha=0.6)
plt.colorbar(sc, label='log(price)')
plt.xlim(28.55, 29.75)
plt.ylim(40.8, 41.3)
plt.xlabel('longitude')
plt.ylabel('latitude')
pass

На карте хорошо видна форма города: европейская и азиатская части, разделённые Босфором, побережье Мраморного моря на юге. Основная масса объявлений — в историческом центре (Фатих, Бейоглу) и вдоль Босфора. Цвет точек показывает цену. Дешёвые и дорогие объявления перемешаны даже в центре (сказывается деление на посуточную и долгосрочную аренду), но есть заметные кластеры более дорогих объявлений: Принцевы острова (Адалар) на юге, Шиле на северо-востоке, отдельные места на побережье. Значит, координаты — полезные признаки, но зависимость от них **нелинейная**: дорого не «на севере» или «на востоке», а в определённых местах. Это аргумент в пользу методов, которые учитывают локальную структуру, — kNN и деревьев.

### Зависимость цены от вместимости и размера

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(x='accommodates', y='log_price', data=train[train['accommodates'] <= 12], ax=axes[0])
sns.boxplot(x='bedrooms', y='log_price', data=train[train['bedrooms'] <= 6], ax=axes[1])
sns.scatterplot(x='amenities_count', y='log_price', data=sample, alpha=0.2, s=8, ax=axes[2])
pass

Чем больше гостей вмещает жильё и чем больше в нём спален, тем оно дороже — зависимость почти монотонная (в логарифмах). Число удобств тоже связано с ценой, но слабее.

### Корреляционная матрица

Посчитаем корреляции между количественными признаками (бинарные признаки тоже числовые).

In [ ]:
corr_mat = train.drop(columns=text_cols + ['long_term']).corr(numeric_only=True)
plt.figure(figsize=(18, 15))
sns.heatmap(corr_mat, square=True, vmin=-1, vmax=1, cmap='coolwarm', xticklabels=True, yticklabels=True)
pass

Корреляция признаков с логарифмом цены:

In [ ]:
corr_mat['log_price'].drop(['price', 'log_price']).sort_values(key=np.abs, ascending=False).head(20)

Сильно коррелированные пары признаков ($|r| > 0.8$):

In [ ]:
corr_pairs = corr_mat.drop(index=['price', 'log_price'], columns=['price', 'log_price'])
corr_pairs = corr_pairs.where(np.triu(np.abs(corr_pairs) > 0.8, k=1)).stack().dropna().sort_values(key=np.abs, ascending=False)
corr_pairs

**Интерпретация.**
- Сильнее всего с логарифмом цены связаны наличие лицензии ($r \approx 0.74$) и минимальный срок аренды ($r \approx -0.72$) — это то самое разделение на краткосрочную и долгосрочную аренду. Сами эти два признака тоже сильно связаны между собой ($r \approx -0.9$).
- Дальше идут признаки размера жилья (`beds`, `accommodates`), число удобств, статус суперхоста, общая ли ванная (у комнат с общей ванной цена ниже).
- Признаки активности (`number_of_reviews_ltm`, `reviews_per_month`, `estimated_occupancy_l365d`) коррелируют с ценой **положительно**, а давность последнего отзыва — отрицательно. Причина та же: активно бронируемые посуточные объявления дороже за ночь, чем долгосрочные, у которых отзывов мало.
- Есть группы почти дублирующих друг друга признаков: `minimum_nights` и его варианты `minimum_minimum_nights`, `maximum_minimum_nights`, `minimum_nights_avg_ntm` (корреляция до 1.0 — это фактически копии); аналогично для `maximum_nights`; `availability_30/60/90`; число объявлений хозяина в разных вариантах подсчёта; оценки `review_scores_*`; число отзывов за разные периоды. Их обработаем в п. 12.
- Корреляция с координатами маленькая — но, как видно по карте, зависимость от них есть, просто нелинейная (коэффициент корреляции Пирсона измеряет только линейную связь).

## 4. Выбросы

Невалидные записи (неправдоподобные цены и размеры) мы уже удалили. Но и среди оставшихся признаков есть признаки с очень тяжёлыми хвостами. Посмотрим на ящики с усами для некоторых из них:

In [ ]:
heavy = ['minimum_nights', 'maximum_nights', 'number_of_reviews', 'reviews_per_month', 'host_listings_count',
         'calculated_host_listings_count', 'accommodates', 'bedrooms', 'amenities_count', 'description_len']
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, col in zip(axes.ravel(), heavy):
    sns.boxplot(y=train[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
pass

Доля выбросов по правилу Тьюки (за пределами $[Q_1-1.5\,{\rm IQR},\ Q_3+1.5\,{\rm IQR}]$) для числовых признаков:

In [ ]:
num_cols = X_train.select_dtypes('number').columns.tolist()
q1 = X_train[num_cols].quantile(0.25)
q3 = X_train[num_cols].quantile(0.75)
iqr = q3 - q1
outlier_share = ((X_train[num_cols] < q1 - 1.5 * iqr) | (X_train[num_cols] > q3 + 1.5 * iqr)).mean()
outlier_share[X_train[num_cols].nunique() > 2].sort_values(ascending=False).head(15)

In [ ]:
q1p, q3p = np.log(y_train).quantile([.25, .75])
iqr_p = q3p - q1p
share_price = (~np.log(y_train).between(q1p - 1.5 * iqr_p, q3p + 1.5 * iqr_p)).mean()
q1p, q3p = y_train.quantile([.25, .75])
iqr_p = q3p - q1p
share_price_raw = (~y_train.between(q1p - 1.5 * iqr_p, q3p + 1.5 * iqr_p)).mean()
share_price_raw, share_price

По цене выбросов по Тьюки около 5%, но после логарифмирования их почти нет — «выбросы» по цене в основном следствие скошенного распределения, а не ошибки. Поэтому дорогие объявления (в пределах 100 000 ₺) мы **не удаляем**: это реальные объявления (виллы, большие квартиры у Босфора), и модель должна уметь их оценивать.

У признаков ситуация такая же: большие значения `number_of_reviews`, `host_listings_count` (есть управляющие компании с сотнями объявлений), `minimum_nights` (объявления для долгосрочной аренды) — правдоподобны, но из-за них распределения очень растянуты. Для метода ближайших соседей это плохо: после стандартизации несколько огромных значений «сжимают» всех остальных к нулю, и признак почти перестаёт влиять на расстояние.

Поэтому строки удалять не будем, а **ограничим (winsorize) значения числовых признаков квантилями уровней 1% и 99%, посчитанными по обучающей выборке**. Например, все `minimum_nights` больше 99-го перцентиля заменятся на значение 99-го перцентиля. Это сделаем отдельным шагом пайплайна, чтобы границы считались только по train.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class QuantileClipper(BaseEstimator, TransformerMixin):
    """Обрезает значения каждого столбца по квантилям, посчитанным на обучающей выборке."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.low_ = X.quantile(self.lower).values
        self.high_ = X.quantile(self.upper).values
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low_, self.high_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

clipper = QuantileClipper().fit(X_train[num_cols])
clipped = pd.DataFrame(clipper.transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
pd.DataFrame({'max до': X_train[num_cols].max(), 'max после': clipped.max()}).loc[heavy]

## 5. Пропущенные значения

In [ ]:
na = X_train.isna().mean().sort_values(ascending=False)
na[na > 0]

Пропуски есть в 20 столбцах:
- `review_scores_*`, `reviews_per_month`, `days_since_first_review`, `days_since_last_review` пропущены одновременно примерно у трети объявлений — **у этих объявлений просто ещё нет отзывов**. Пропуск здесь сам по себе информативен (новое объявление). Поэтому, кроме заполнения медианой, добавим бинарный признак-индикатор «значение было пропущено» (`add_indicator=True` в `SimpleImputer`).
- `beds`, `bathrooms`, `bedrooms`, `bath_shared` — хозяин не указал. Заполняем медианой (индикатор тоже добавится).
- `has_availability` — почти все объявления имеют значение 1, заполняем медианой.
- `description` и `host_about` — тексты. Пропуск означает, что текста нет; заменим на пустую строку (длина текста уже посчитана и равна 0).

В категориальных признаках пропусков нет, но на всякий случай (для новых данных) поставим заполнение самым частым значением.

Медианы, как и всё остальное, считаются по обучающей выборке внутри пайплайна.

In [ ]:
X_train[cat_cols].isna().sum()

## 6. Категориальные признаки

- Бинарные признаки (`host_is_superhost`, `host_identity_verified`, наличие удобств и т.п.) уже закодированы как 0/1.
- `room_type` (4 значения) и `neighbourhood_cleansed` (39 районов) — номинальные признаки без порядка, применим **one-hot encoding** (бинаризацию).
- `property_type` имеет 90 значений, многие встречаются по несколько раз. Редкие значения (встречающиеся в обучающей выборке менее 50 раз) объединим в одну категорию «прочее» (`min_frequency=50`), иначе получится много почти пустых dummy-столбцов. Этот же приём применим к редким районам.
- Если в тесте попадётся значение, которого не было в обучающей выборке, оно тоже попадёт в «прочее» (`handle_unknown='infrequent_if_exist'`).

In [ ]:
train['property_type'].value_counts().head(25)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50, sparse_output=False)
enc.fit(X_train[cat_cols])
[len(c) for c in enc.categories_], len(enc.get_feature_names_out())

In [ ]:
enc.get_feature_names_out()[:20]

## 7. Нормализация

Нормализация **необходима**, так как основной метод — k ближайших соседей, а он основан на расстояниях между объектами. Признаки имеют совершенно разные масштабы: широта и долгота меняются в пределах десятых долей, `accommodates` — от 1 до 16, `days_since_first_review` — до нескольких тысяч, длина описания — до тысячи символов. Без нормализации расстояние определялось бы почти только признаками с большим разбросом.

Будем использовать стандартизацию (вычитаем среднее и делим на стандартное отклонение, посчитанные по обучающей выборке) — `StandardScaler`. Применяем её после ограничения выбросов, иначе среднее и разброс считались бы с учётом выбросов.

Соберём всю предобработку в один `Pipeline`, как в конце лекции: так все параметры (квантили, медианы, категории, средние) гарантированно считаются только по обучающей выборке и одинаково применяются к тестовой.

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def make_preprocessor(num_features):
    return ColumnTransformer([
        ('num', make_pipeline(
            QuantileClipper(0.01, 0.99),
            SimpleImputer(strategy='median', add_indicator=True),
            StandardScaler()), num_features),
        ('cat', make_pipeline(
            SimpleImputer(strategy='most_frequent'),
            OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50, sparse_output=False),
            StandardScaler()), cat_cols),
    ])

preprocessor = make_preprocessor(num_cols)
X_train_prep = preprocessor.fit_transform(X_train)
X_train_prep.shape

In [ ]:
pd.DataFrame(X_train_prep).describe().T[['mean', 'std']].describe()

После предобработки у каждого признака среднее 0 и стандартное отклонение 1 (кроме константных индикаторов). Dummy-признаки тоже стандартизированы, чтобы они имели такой же «вес» в расстоянии, как и числовые.

Итого получилось 135 признаков: числовые, индикаторы пропусков и dummy-переменные.

## 8. Метод k ближайших соседей

**Почему kNN.** Цена жилья во многом определяется тем, *где* оно и *какое* оно, — а похожие объявления (рядом на карте, с той же вместимостью, тем же типом жилья) стоят похоже. Это ровно та идея, на которой основан метод ближайших соседей: прогноз — среднее значение цены по k наиболее похожим объявлениям обучающей выборки. Метод
- не предполагает какую-то форму зависимости (в отличие от линейной регрессии) — а мы видели, что зависимость от координат явно нелинейная;
- прост и хорошо интерпретируется («такая цена у похожих объявлений»);
- имеет один основной гиперпараметр k, который подберём по кросс-валидации.

Минусы: чувствителен к масштабу признаков (поэтому нормализация) и к «мусорным» признакам — все признаки одинаково участвуют в расстоянии. Сравним его с другими методами в п. 10.

Как обсуждалось выше, обучаемся на логарифме цены, а предсказание переводим обратно экспонентой.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

knn = Pipeline([
    ('prep', make_preprocessor(num_cols)),
    ('fit', KNeighborsRegressor(n_neighbors=5)),
])
knn.fit(X_train, y_train_log)

## 9. Ошибки на обучающей и тестовой выборках, подбор k

Будем считать:
- **RMSE** и **MAE** для логарифма цены (RMSE = 0.5 по логарифму примерно означает «типичная ошибка в $e^{0.5}\approx 1.65$ раза»);
- **$R^2$** (коэффициент детерминации) для логарифма цены — какая доля разброса объясняется моделью;
- **MAE** и **медианную абсолютную ошибку** в лирах (после обратного преобразования) и **MAPE** — среднюю относительную ошибку — чтобы ошибку можно было интерпретировать «в деньгах».

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error, r2_score, mean_absolute_percentage_error

def metrics(y_true_log, y_pred_log):
    y_true, y_pred = np.exp(y_true_log), np.exp(y_pred_log)
    return {
        'RMSE (log)': mean_squared_error(y_true_log, y_pred_log) ** 0.5,
        'MAE (log)': mean_absolute_error(y_true_log, y_pred_log),
        'R2 (log)': r2_score(y_true_log, y_pred_log),
        'MAE, ₺': mean_absolute_error(y_true, y_pred),
        'MedAE, ₺': median_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred),
    }

results = {}

def evaluate(name, model, Xtr=None, Xte=None):
    Xtr = X_train if Xtr is None else Xtr
    Xte = X_test if Xte is None else Xte
    m_train = metrics(y_train_log, model.predict(Xtr))
    m_test = metrics(y_test_log, model.predict(Xte))
    results[name] = {**{k + ' train': v for k, v in m_train.items()}, **{k + ' test': v for k, v in m_test.items()}}
    return pd.DataFrame({'train': m_train, 'test': m_test}).round(3)

evaluate('kNN, k=5', knn)

Для начала — базовый уровень: «модель», которая всегда предсказывает медианную цену обучающей выборки. С ней будем сравнивать остальные модели.

In [ ]:
from sklearn.dummy import DummyRegressor
baseline = DummyRegressor(strategy='median').fit(X_train, y_train_log)
evaluate('Константа (медиана)', baseline)

При k = 5 ошибка на обучающей выборке заметно меньше, чем на тестовой: объект обучающей выборки сам входит в число своих соседей, поэтому на train ошибка занижена. Но и на тесте kNN гораздо лучше константы: $R^2 \approx 0.75$, то есть модель объясняет три четверти разброса логарифма цены (константа — ноль).

### Подбор числа соседей

Подберём k по 5-кратной кросс-валидации на обучающей выборке (тестовая выборка в подборе не участвует). Заодно проверим, что лучше — простое среднее соседей или среднее с весами, обратно пропорциональными расстоянию (`weights='distance'`).

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)
k_arr = np.array([1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 70, 100])

grid = GridSearchCV(knn, {'fit__n_neighbors': k_arr, 'fit__weights': ['uniform', 'distance']},
                    scoring='neg_root_mean_squared_error', cv=cv, return_train_score=True, n_jobs=-1)
grid.fit(X_train, y_train_log)
grid.best_params_, -grid.best_score_

In [ ]:
cvres = pd.DataFrame(grid.cv_results_)
uni = cvres[cvres['param_fit__weights'] == 'uniform']
dist = cvres[cvres['param_fit__weights'] == 'distance']

plt.figure(figsize=(10, 5))
plt.plot(k_arr, -uni['mean_test_score'], 'r.-', label='CV, uniform')
plt.fill_between(k_arr, -uni['mean_test_score'] - uni['std_test_score'], -uni['mean_test_score'] + uni['std_test_score'], alpha=0.15, color='r')
plt.plot(k_arr, -uni['mean_train_score'], 'b.-', label='train, uniform')
plt.plot(k_arr, -dist['mean_test_score'], 'g.--', label='CV, distance')
plt.xscale('log')
plt.xlabel('k')
plt.ylabel('RMSE (log price)')
plt.legend()
pass

In [ ]:
cvres.pivot_table(index='param_fit__n_neighbors', columns='param_fit__weights', values='mean_test_score').mul(-1).round(4)

Картина типичная для kNN:
- при **k = 1** ошибка на обучающей выборке нулевая (каждый объект — сам себе ближайший сосед), а на кросс-валидации — самая большая: модель **переобучается**, запоминает шум;
- с ростом k ошибка на обучающей выборке растёт, а на валидации сначала падает, а потом начинает медленно расти — при слишком большом k прогноз усредняется по слишком разным объявлениям (**недообучение**);
- минимум ошибки на кросс-валидации достигается при k = 5–10. Взвешивание по расстоянию даёт немного меньшую ошибку на CV, чем простое среднее, при любом k.

Обучим модель с оптимальными параметрами на всей обучающей выборке и посчитаем ошибки:

In [ ]:
best_k = grid.best_params_['fit__n_neighbors']
best_w = grid.best_params_['fit__weights']
knn_best = grid.best_estimator_
evaluate(f'kNN, k={best_k}, {best_w}', knn_best)

Ошибка на тестовой выборке уменьшилась по сравнению с k = 5. Ошибка на train при `weights='distance'` близка к нулю (объект совпадает сам с собой, расстояние 0, вес бесконечный) — поэтому на эту цифру ориентироваться нельзя, честная оценка — тестовая ошибка и кросс-валидация.

**Как понимать ошибку:** RMSE по логарифму ≈ 0.55 — типичный прогноз отличается от реальной цены в $e^{0.55}\approx 1.7$ раза в ту или другую сторону; при этом медианная абсолютная ошибка — около 500 ₺ при медианной цене ≈ 2 900 ₺ (то есть у половины объявлений ошибка меньше 500 ₺). Большой вклад в RMSE дают сравнительно редкие объявления, на которых модель сильно ошибается.

Посмотрим на прогнозы против реальных значений на тестовой выборке:

In [ ]:
y_test_pred_log = knn_best.predict(X_test)
plt.figure(figsize=(7, 7))
plt.scatter(y_test_log, y_test_pred_log, alpha=0.1, s=6)
lims = [y_test_log.min(), y_test_log.max()]
plt.plot(lims, lims, 'r--')
plt.xlabel('истинный log(price)')
plt.ylabel('предсказанный log(price)')
pass

Точки вытянуты вдоль диагонали, но заметно «прижаты» к среднему: дешёвые объявления модель переоценивает, дорогие — недооценивает. Это свойство усреднения по соседям — прогноз kNN никогда не выходит за пределы цен соседей.

### Матрица рассогласования

Матрица рассогласования определена для задачи классификации, у нас регрессия. Но для наглядности можно разбить цены на 4 ценовые категории по квартилям обучающей выборки и посмотреть, насколько часто модель «попадает» в правильную категорию:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

bins = np.r_[-np.inf, np.quantile(y_train_log, [0.25, 0.5, 0.75]), np.inf]
labels = ['дешёвые', 'ниже среднего', 'выше среднего', 'дорогие']
cls_true = pd.cut(y_test_log, bins, labels=labels)
cls_pred = pd.cut(y_test_pred_log, bins, labels=labels)
cm = confusion_matrix(cls_true, cls_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap='Blues', values_format='d')
plt.xticks(rotation=30)
pass

In [ ]:
codes_true = np.asarray(cls_true.cat.codes)
codes_pred = np.asarray(pd.Categorical(cls_pred, categories=labels).codes)
(codes_true == codes_pred).mean(), (np.abs(codes_true - codes_pred) <= 1).mean()

Большая часть объектов лежит на диагонали и рядом с ней: в «правильную» ценовую категорию (из четырёх) модель попадает примерно в 69% случаев, а ошибается больше чем на одну категорию лишь в ~3% случаев. Ошибки «дешёвое ↔ дорогое» почти отсутствуют. Чаще всего модель путает соседние категории в середине распределения («ниже среднего» ↔ «выше среднего»).

## 10. Другие методы

Сравним kNN с методами, которые были на лекциях:
- **линейная регрессия** (с небольшой $L_2$-регуляризацией — Ridge — из-за коррелированных признаков);
- **случайный лес**;
- **градиентный бустинг** (`HistGradientBoostingRegressor` — быстрая реализация бустинга над деревьями).

Деревьям нормализация не нужна, но мы используем ту же предобработку для единообразия (на деревья она не влияет).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

ridge = Pipeline([('prep', make_preprocessor(num_cols)), ('fit', Ridge(alpha=1.0))])
ridge.fit(X_train, y_train_log)
evaluate('Линейная регрессия (Ridge)', ridge)

In [ ]:
rf = Pipeline([('prep', make_preprocessor(num_cols)),
               ('fit', RandomForestRegressor(n_estimators=300, min_samples_leaf=2, max_features=0.3, n_jobs=-1, random_state=42))])
rf.fit(X_train, y_train_log)
evaluate('Случайный лес', rf)

In [ ]:
gb = Pipeline([('prep', make_preprocessor(num_cols)),
               ('fit', HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.05, max_leaf_nodes=63,
                                                     min_samples_leaf=20, l2_regularization=1.0, random_state=42))])
gb.fit(X_train, y_train_log)
evaluate('Градиентный бустинг', gb)

- **Линейная регрессия** на тесте немного лучше kNN по RMSE ($R^2 \approx 0.77$), причём ошибки на train и test почти совпадают — модель простая и не переобучается. Но медианная ошибка у неё больше, чем у kNN: линейная модель хуже описывает нелинейные эффекты (координаты, взаимодействие типа аренды и размера жилья).
- **Случайный лес** и **градиентный бустинг** заметно лучше: RMSE по логарифму ≈ 0.45 против ≈ 0.55 у kNN, $R^2 \approx 0.84$. Деревья сами выбирают, какие признаки важны, и не страдают от того, что в расстоянии kNN все 135 признаков весят одинаково. У обоих ансамблей ошибка на train заметно ниже, чем на test, — есть переобучение, но тестовая ошибка всё равно лучшая.

### Важность признаков (случайный лес)

In [ ]:
feat_names = rf.named_steps['prep'].get_feature_names_out()
importances = pd.Series(rf.named_steps['fit'].feature_importances_, index=feat_names).sort_values(ascending=False)
plt.figure(figsize=(8, 8))
sns.barplot(x=importances.head(25), y=importances.head(25).index, orient='h')
pass

Как и ожидалось после анализа данных, самые важные признаки — **минимальный срок аренды** (во всех вариантах) и **наличие лицензии**, то есть разделение на посуточную и долгосрочную аренду. Дальше идут размер жилья (`accommodates`, `bedrooms`, `bathrooms`, `beds`), координаты, количество удобств, тип жилья (комната или целиком), длина описания. Признаки, связанные с хозяином и оценками, менее важны.

### Добавляем текстовые признаки

До сих пор из текстов мы использовали только их длину. Попробуем учесть содержание названия и описания: построим **TF-IDF** представление (каждый текст → вектор весов слов и пар слов), а затем сожмём его до 50 компонент с помощью усечённого SVD (аналог метода главных компонент для разреженных матриц). Слова вроде «villa», «bosphorus view», «luxury», «room», «hostel» должны нести информацию о цене.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import FunctionTransformer

def join_text(X):
    X = pd.DataFrame(X)
    return (X['name'].fillna('') + ' ' + X['description'].fillna('')).values

text_pipe = make_pipeline(
    FunctionTransformer(join_text),
    TfidfVectorizer(min_df=5, max_features=30000, ngram_range=(1, 2), sublinear_tf=True),
    TruncatedSVD(n_components=50, random_state=42),
    StandardScaler())

def make_preprocessor_text(num_features):
    return ColumnTransformer([
        ('num', make_pipeline(QuantileClipper(0.01, 0.99), SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), num_features),
        ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                              OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50, sparse_output=False),
                              StandardScaler()), cat_cols),
        ('text', text_pipe, ['name', 'description']),
    ])

gb_text = Pipeline([('prep', make_preprocessor_text(num_cols)),
                    ('fit', HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.05, max_leaf_nodes=63,
                                                          min_samples_leaf=20, l2_regularization=1.0, random_state=42))])
gb_text.fit(X_train, y_train_log)
evaluate('Градиентный бустинг + TF-IDF', gb_text)

In [ ]:
knn_text = Pipeline([('prep', make_preprocessor_text(num_cols)),
                     ('fit', KNeighborsRegressor(n_neighbors=best_k, weights=best_w))])
knn_text.fit(X_train, y_train_log)
evaluate(f'kNN, k={best_k} + TF-IDF', knn_text)

Текстовые признаки в таком виде **качество не улучшили**: у бустинга ошибка почти не изменилась (0.445 → 0.452), у kNN — тоже стала чуть хуже. Видимо, основная информация из текстов (тип жилья, число спален, расположение) уже есть в структурированных признаках, а 50 дополнительных SVD-компонент для kNN — это скорее шум в расстоянии.

Слова, которые сильнее всего связаны с ценой (коэффициенты линейной регрессии на TF-IDF признаках), — просто чтобы убедиться, что тексты действительно содержат информацию о цене:

In [ ]:
tfidf = TfidfVectorizer(min_df=5, max_features=30000, ngram_range=(1, 2), sublinear_tf=True)
T_train = tfidf.fit_transform(join_text(X_train))
lin_text = Ridge(alpha=3.0).fit(T_train, y_train_log - y_train_log.mean())
coefs = pd.Series(lin_text.coef_, index=tfidf.get_feature_names_out()).sort_values()
pd.DataFrame({'дешевле': coefs.head(15).index, 'дороже': coefs.tail(15).index[::-1]})

Слова получились осмысленными: «room», «private room», турецкие «oda» (комната), «özel oda» (отдельная комната), «kiralık» (сдаётся), а также «students», «metrobus», «weekly» — это признаки дешёвых комнат и долгосрочной аренды. Слова, связанные с дорогими объявлениями: «villa», «pool», «havuzlu» (с бассейном), «luxury», «bosphorus», «3br»/«2br», «family», «galata». То есть тексты действительно содержат информацию о цене, просто она в основном дублирует остальные признаки.

## 11. Несбалансированность классов

У нас задача регрессии, классов нет, поэтому этот пункт неприменим. Аналог проблемы в регрессии — сильно скошенное распределение целевой переменной (мало дорогих объявлений, которые сильно влияют на MSE). С этим мы боролись, перейдя к логарифму цены.

## 12. Исключение коррелированных признаков

Зачем это нужно:
- для **kNN** все признаки одинаково входят в расстояние. Если одну и ту же информацию несут несколько признаков (например, четыре почти одинаковых варианта `minimum_nights`), она учитывается в расстоянии многократно и «перевешивает» остальные признаки;
- для **линейной регрессии** сильно коррелированные признаки (мультиколлинеарность) делают коэффициенты неустойчивыми и плохо интерпретируемыми;
- меньше признаков — быстрее обучение и предсказание, модель проще.

Будем жадно удалять признаки: перебираем пары с $|r|$ больше порога (корреляции считаем по обучающей выборке) и из каждой пары удаляем тот признак, который слабее коррелирует с логарифмом цены. Попробуем два порога: 0.95 (удаляются только практически копии) и 0.8.

In [ ]:
corr_num = train[num_cols + ['log_price']].corr()
target_corr = corr_num['log_price'].abs()

def correlated_to_drop(threshold):
    to_drop = set()
    for (a, b), r in corr_pairs[corr_pairs.abs() > threshold].items():
        if a in to_drop or b in to_drop:
            continue
        to_drop.add(a if target_corr[a] < target_corr[b] else b)
    return sorted(to_drop)

drop_95 = correlated_to_drop(0.95)
drop_80 = correlated_to_drop(0.8)
print(len(drop_95), drop_95)
print(len(drop_80), drop_80)

In [ ]:
for thr, to_drop in [(0.95, drop_95), (0.8, drop_80)]:
    cols = [c for c in num_cols if c not in to_drop]
    knn_red = Pipeline([('prep', make_preprocessor(cols)), ('fit', KNeighborsRegressor())])
    grid_red = GridSearchCV(knn_red, {'fit__n_neighbors': [5, 7, 10, 15], 'fit__weights': ['distance']},
                            scoring='neg_root_mean_squared_error', cv=cv, n_jobs=-1)
    grid_red.fit(X_train, y_train_log)
    k_red = grid_red.best_params_['fit__n_neighbors']
    print(f'|r| > {thr}: осталось {len(cols)} числовых признаков, лучшее k = {k_red}, RMSE на CV = {-grid_red.best_score_:.4f}')
    evaluate(f'kNN без коррелированных (|r|>{thr}), k={k_red}', grid_red.best_estimator_)
    ridge_red = Pipeline([('prep', make_preprocessor(cols)), ('fit', Ridge(alpha=1.0))]).fit(X_train, y_train_log)
    evaluate(f'Ridge без коррелированных (|r|>{thr})', ridge_red)

pd.DataFrame(results).T.loc[lambda t: t.index.str.contains('kNN, k=7, distance|Ridge|коррелир'),
                            ['RMSE (log) test', 'R2 (log) test', 'MedAE, ₺ test']].round(3)

Результат получился не таким, как ожидалось: **удаление коррелированных признаков не улучшило kNN, а немного ухудшило** (RMSE на тесте 0.549 → ≈0.57 при пороге 0.95 и ≈0.58 при пороге 0.8). У линейной регрессии при пороге 0.95 качество не изменилось, а при пороге 0.8 стало хуже.

Объяснение:
- При пороге 0.8 удаляется и сам `minimum_nights` (он коррелирует с `has_license` с $r\approx-0.9$, а `has_license` чуть сильнее связан с ценой), и все его варианты. Но `minimum_nights` несёт дополнительную информацию (среди объявлений без лицензии бывают и посуточные, и наоборот), и её потеря ухудшает и kNN, и линейную модель.
- При пороге 0.95 удаляются только почти точные копии (`minimum_minimum_nights` = `minimum_nights`, и т.п.), и для линейной модели ничего не меняется, как и должно быть. А вот kNN становится чуть хуже, потому что **копии самого важного признака случайно работали как увеличение его веса** в расстоянии: четыре варианта `minimum_nights` давали этому признаку в 4 раза больший вклад, а он действительно самый важный (см. важность признаков у случайного леса).

Вывод: для kNN проблема не столько в коррелированности признаков, сколько в том, что все признаки в расстоянии имеют одинаковый вес, а полезность у них очень разная. Улучшать kNN стоило бы через отбор признаков по важности или подбор весов признаков, а не только через удаление коррелированных.

## 13. Сравнение моделей и выводы

In [ ]:
res = pd.DataFrame(results).T
res[['RMSE (log) train', 'RMSE (log) test', 'R2 (log) train', 'R2 (log) test', 'MedAE, ₺ test', 'MAPE test']].round(3).sort_values('RMSE (log) test')

In [ ]:
plt.figure(figsize=(9, 5))
r = res['RMSE (log) test'].sort_values()
sns.barplot(x=r, y=r.index, orient='h')
plt.xlabel('RMSE (log price) на тестовой выборке')
pass

### Выводы

1. Решалась задача **регрессии**: предсказание цены за ночь объявления Airbnb в Стамбуле по ~70 исходным признакам разных типов (числовые, категориальные, бинарные, даты, список удобств, тексты). После очистки осталось ≈ 23.5 тыс. объявлений, из них 75% — обучающая выборка.
2. **Главная особенность данных** — смесь двух рынков: посуточной аренды (в основном с лицензией, медиана ≈ 4 300 ₺ за ночь) и фактически долгосрочной аренды с минимальным сроком 100 ночей (без лицензии, медиана ≈ 690 ₺ за ночь). Из-за этого распределение цены двугорбое, а `minimum_nights` и `has_license` — самые важные признаки. Дальше по важности идут размер жилья (вместимость, спальни, ванные), координаты, тип жилья и число удобств.
3. Цена сильно скошена вправо, поэтому модели обучались на **логарифме цены**. Невалидные записи (цена < 150 ₺ или > 100 000 ₺, > 20 спален и т.п.) удалены; выбросы в признаках не удалялись, а **ограничивались квантилями 1%/99%** обучающей выборки. Пропуски заполнялись медианами с добавлением признаков-индикаторов пропуска (пропуск оценок означает «нет отзывов»). Категориальные признаки — one-hot с объединением редких категорий. Все признаки стандартизированы — это обязательно для kNN.
4. **Метод k ближайших соседей**: оптимальное число соседей по кросс-валидации — **k = 7** с весами, обратными расстоянию. При k = 1 модель переобучается (ошибка на train 0), при больших k — недообучается. На тесте: RMSE(log) ≈ 0.55, $R^2 \approx 0.77$, медианная абсолютная ошибка ≈ 530 ₺ (константный прогноз даёт RMSE(log) ≈ 1.15 и $R^2 \approx 0$).
5. **Сравнение методов** (по RMSE логарифма цены на тесте): градиентный бустинг (≈ 0.445, $R^2 \approx 0.85$) ≈ случайный лес (≈ 0.457) < линейная регрессия (≈ 0.543) ≈ kNN (≈ 0.549) ≪ константа (1.15). Ансамбли деревьев лучше, потому что сами отбирают важные признаки и учитывают нелинейности и взаимодействия; kNN проигрывает из-за того, что все признаки входят в расстояние с одинаковым весом.
6. **Текстовые признаки** (TF-IDF названия и описания) содержат осмысленную информацию о цене («villa», «pool», «bosphorus» ↔ «room», «students», «weekly»), но заметного улучшения не дали — эта информация в основном дублирует структурированные признаки.
7. **Удаление коррелированных признаков** для kNN неожиданно немного ухудшило результат: дубли самого важного признака `minimum_nights` фактически увеличивали его вес в расстоянии. Для kNN полезнее было бы отбирать признаки по важности или подбирать их веса.
8. Несбалансированности классов в задаче регрессии нет; её аналог — скошенность целевой переменной — учтён логарифмированием.